# Analyse simplifiée des vues LLM et des poids Black-Litterman

Ce notebook garde uniquement les analyses utiles pour documenter le style des vues LLM et la dynamique des portefeuilles :

1. boxplots globaux de dispersion des vues par LLM;
2. vues moyennes dans le temps;
3. proportion de vues positives dans le temps;
4. alignement des vues avec les régimes de marché;
5. variation absolue moyenne des vues dans le temps;
6. histogrammes cumulatifs des poids des portefeuilles.

Aucune exportation CSV n'est faite dans cette version.


In [ ]:
import json
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:.6f}")

# ============================
# Configuration
# ============================

START = "2015-01-01"
END = "2025-06-30"

PORTFOLIO_RISK_AVERSION = 1
TAU = 0.001

# Le notebook est supposé être exécuté depuis le dossier analyses/.
# Change BASE_DIR à Path(".") si tu l'exécutes depuis la racine du projet.
BASE_DIR = Path("..")

RESPONSES_DIR = BASE_DIR / "responses"
DATA_PATH = BASE_DIR / "data" / "filtered_sp25_data.csv"


def format_portfolio_risk_aversion(value: float | str) -> str:
    try:
        return f"{float(value):g}"
    except Exception:
        return str(value).strip().replace(" ", "_")


RESULTS_DIR = BASE_DIR / "results" / f"portfolio_risk_aversion_{format_portfolio_risk_aversion(PORTFOLIO_RISK_AVERSION)}"

# Modèles analysés. Le nom à gauche doit correspondre au préfixe des fichiers JSON et CSV.
MODELS = {
    "gpt54mini": "GPT-5.4-mini",
    "gemma3": "Gemma3",
    "qwen": "Qwen2.5",
    "llama": "Llama3.2",
}

START_DT = pd.to_datetime(START)
END_DT = pd.to_datetime(END)

print("Dossier responses:", RESPONSES_DIR.resolve())
print("Dossier results:", RESULTS_DIR.resolve())
print("Fichier de données:", DATA_PATH.resolve())
print("Période:", START, "à", END)
print("Portfolio risk aversion:", PORTFOLIO_RISK_AVERSION)
print("Tau:", TAU)


## 1. Chargement des vues LLM

Les fichiers attendus ont la forme :

```text
responses/{model}_{YYYY-MM-DD}_{YYYY-MM-DD}.json
```

Chaque fichier contient les vues mensuelles par titre pour un modèle donné.


In [ ]:
def extract_period_from_filename(path: str | Path) -> tuple[pd.Timestamp, pd.Timestamp, pd.Timestamp]:
    name = Path(path).stem
    match = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})", name)

    if not match:
        return pd.NaT, pd.NaT, pd.NaT

    start_date = pd.to_datetime(match.group(1), errors="coerce")
    end_date = pd.to_datetime(match.group(2), errors="coerce")

    # Convention du pipeline : la vue est associée au début du mois de rééquilibrage.
    rebalance_date = start_date

    return start_date, end_date, rebalance_date


def parse_expected_return_values(x) -> list[float]:
    if x is None:
        return []

    if isinstance(x, dict):
        for key in ["expected_return", "Expected Return", "expected_returns", "return", "predicted_return", "forecast_return", "view"]:
            if key in x:
                return parse_expected_return_values(x[key])
        return []

    if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        out = []
        for item in x:
            out.extend(parse_expected_return_values(item))
        return out

    if isinstance(x, str):
        s = x.strip().replace(",", ".")
        is_percent = "%" in s
        s = s.replace("%", "")
        val = pd.to_numeric(s, errors="coerce")

        if pd.isna(val):
            return []

        val = float(val)
        if is_percent or abs(val) > 1:
            val = val / 100

        return [val]

    val = pd.to_numeric(x, errors="coerce")
    if pd.isna(val):
        return []

    return [float(val)]


def read_response_file(path: str | Path, model_key: str, model_name: str) -> pd.DataFrame:
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    start_date, end_date, rebalance_date = extract_period_from_filename(path)
    rows = []

    # Format principal du pipeline : {ticker: {"expected_return": [...]}, ...}
    if isinstance(data, dict):
        items = data.items()
    elif isinstance(data, list):
        items = [(obj.get("ticker", obj.get("symbol", np.nan)), obj) for obj in data if isinstance(obj, dict)]
    else:
        items = []

    metadata_keys = {"model", "start_date", "end_date", "date", "prompt", "metadata"}

    for ticker, obj in items:
        if str(ticker).lower() in metadata_keys:
            continue

        vals = parse_expected_return_values(obj)
        if not vals:
            continue

        rows.append({
            "model_key": model_key,
            "model": model_name,
            "ticker": str(ticker).strip().upper(),
            "expected_return": float(np.mean(vals)),
            "n_raw_values": len(vals),
            "file": path.name,
            "period_start": start_date,
            "period_end": end_date,
            "rebalance_date": rebalance_date,
        })

    return pd.DataFrame(rows)


def load_all_views(models: dict[str, str], responses_dir: Path) -> pd.DataFrame:
    pieces = []

    for model_key, model_name in models.items():
        files = sorted(glob.glob(str(responses_dir / f"{model_key}_*.json")))

        if not files:
            print(f"Aucun fichier trouvé pour {model_name} ({model_key}).")
            continue

        print(f"{model_name}: {len(files)} fichier(s) trouvé(s).")

        for fp in files:
            d = read_response_file(fp, model_key, model_name)
            if not d.empty:
                pieces.append(d)

    if not pieces:
        raise FileNotFoundError("Aucune vue LLM n'a été chargée. Vérifie RESPONSES_DIR et MODELS.")

    views = pd.concat(pieces, ignore_index=True)
    views["rebalance_date"] = pd.to_datetime(views["rebalance_date"], errors="coerce")
    views["expected_return"] = pd.to_numeric(views["expected_return"], errors="coerce")

    views = views.dropna(subset=["rebalance_date", "expected_return"]).copy()
    views = views[(views["rebalance_date"] >= START_DT) & (views["rebalance_date"] <= END_DT)].copy()

    if views.empty:
        raise ValueError("Aucune vue ne reste après le filtre START-END.")

    views["positive_view"] = views["expected_return"] > 0
    views["year_month"] = views["rebalance_date"].dt.to_period("M").astype(str)

    return views.sort_values(["model", "rebalance_date", "ticker"]).reset_index(drop=True)


views = load_all_views(MODELS, RESPONSES_DIR)

print("\nDimensions:", views.shape)
print("Période observée:", views["rebalance_date"].min(), "à", views["rebalance_date"].max())
print("Modèles:", sorted(views["model"].unique()))
display(views.head())


## 2. Boxplots globaux de dispersion des vues par LLM

Chaque boîte résume la distribution complète des vues d'un modèle sur toute la période.


In [ ]:
models = sorted(views["model"].unique())
box_data = [views.loc[views["model"] == m, "expected_return"].dropna().values for m in models]

plt.figure(figsize=(10, 6))
plt.boxplot(box_data, labels=models, showfliers=False)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Dispersion globale des vues par LLM")
plt.ylabel("Vue LLM / rendement attendu")
plt.xlabel("Modèle")
plt.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 3. Vues moyennes dans le temps

In [ ]:
monthly_views = (
    views
    .groupby(["rebalance_date", "model"])
    .agg(
        mean_view=("expected_return", "mean"),
        positive_view_ratio=("positive_view", "mean"),
        n_views=("expected_return", "count"),
    )
    .reset_index()
)

mean_views_ts = (
    monthly_views
    .pivot(index="rebalance_date", columns="model", values="mean_view")
    .sort_index()
)

plt.figure(figsize=(12, 5))
for col in mean_views_ts.columns:
    plt.plot(mean_views_ts.index, mean_views_ts[col], label=col)

plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Vue moyenne dans le temps")
plt.ylabel("Vue moyenne")
plt.xlabel("Date")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

display(mean_views_ts.tail())


## 4. Proportion de vues positives dans le temps

In [ ]:
positive_ratio_ts = (
    monthly_views
    .pivot(index="rebalance_date", columns="model", values="positive_view_ratio")
    .sort_index()
)

plt.figure(figsize=(12, 5))
for col in positive_ratio_ts.columns:
    plt.plot(positive_ratio_ts.index, positive_ratio_ts[col], label=col)

plt.axhline(0.5, linestyle="--", linewidth=1)
plt.ylim(-0.05, 1.05)
plt.title("Proportion de vues positives dans le temps")
plt.ylabel("Proportion de vues positives")
plt.xlabel("Date")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

display(positive_ratio_ts.tail())


## 5. Alignement des vues avec les régimes de marché

Le régime de marché est reconstruit localement à partir du fichier `filtered_sp25_data.csv`, en utilisant un rendement mensuel pondéré par `market_equity`.


In [ ]:
def load_market_returns_from_dataset(path: str | Path) -> pd.Series:
    data = pd.read_csv(path, low_memory=False)

    required = {"date", "stock_ret", "market_equity"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes pour construire le régime de marché: {sorted(missing)}")

    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["stock_ret"] = pd.to_numeric(data["stock_ret"], errors="coerce")
    data["market_equity"] = pd.to_numeric(data["market_equity"], errors="coerce")

    data = data.dropna(subset=["date", "stock_ret", "market_equity"]).copy()
    data = data[data["market_equity"] > 0].copy()
    data["month"] = data["date"].dt.to_period("M").dt.to_timestamp()

    def cap_weighted_return(g: pd.DataFrame) -> float:
        w = g["market_equity"] / g["market_equity"].sum()
        return float((w * g["stock_ret"]).sum())

    market_returns = data.groupby("month").apply(cap_weighted_return).sort_index()
    market_returns.name = "market_return"

    return market_returns[(market_returns.index >= START_DT) & (market_returns.index <= END_DT)]


market_returns = load_market_returns_from_dataset(DATA_PATH)

market_regime = market_returns.to_frame()
market_regime["market_direction"] = np.where(market_regime["market_return"] >= 0, 1, -1)
market_regime["regime"] = np.where(market_regime["market_return"] >= 0, "Haussier", "Baissier")

view_regime = monthly_views.copy()
view_regime["month"] = pd.to_datetime(view_regime["rebalance_date"]).dt.to_period("M").dt.to_timestamp()
view_regime = view_regime.merge(market_regime.reset_index(), on="month", how="left")

view_regime["view_direction"] = np.where(view_regime["positive_view_ratio"] >= 0.5, 1, -1)
view_regime["aligned"] = view_regime["view_direction"] == view_regime["market_direction"]

alignment_summary = (
    view_regime
    .dropna(subset=["market_return"])
    .groupby("model")
    .agg(
        n_months=("month", "count"),
        avg_positive_ratio=("positive_view_ratio", "mean"),
        alignment_rate=("aligned", "mean"),
        corr_positive_ratio_market_return=("positive_view_ratio", lambda x: x.corr(view_regime.loc[x.index, "market_return"])),
        avg_market_return_when_views_bullish=("market_return", lambda x: x[view_regime.loc[x.index, "positive_view_ratio"] >= 0.5].mean()),
        avg_market_return_when_views_bearish=("market_return", lambda x: x[view_regime.loc[x.index, "positive_view_ratio"] < 0.5].mean()),
    )
    .reset_index()
)

display(alignment_summary)

for model in sorted(view_regime["model"].dropna().unique()):
    d = view_regime[view_regime["model"] == model].dropna(subset=["market_return"]).sort_values("month")

    if d.empty:
        continue

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.plot(d["month"], d["positive_view_ratio"], label="Proportion de vues positives")
    ax1.axhline(0.5, linestyle="--", linewidth=1)
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_ylabel("Proportion de vues positives")
    ax1.set_xlabel("Date")
    ax1.grid(True, linestyle="--", alpha=0.4)

    ax2 = ax1.twinx()
    ax2.bar(d["month"], d["market_return"], width=20, alpha=0.25, label="Rendement du marché")
    ax2.set_ylabel("Rendement mensuel du marché")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()

    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")
    plt.title(f"Alignement vues-régime de marché - {model}")
    fig.tight_layout()
    plt.show()


## 6. Variation absolue moyenne des vues dans le temps

Cette mesure suit l'ampleur moyenne des changements de vues d'un mois à l'autre, pour les mêmes titres.


In [ ]:
def compute_mean_abs_view_change(views_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model, d in views_df.groupby("model"):
        wide = (
            d.pivot_table(
                index="rebalance_date",
                columns="ticker",
                values="expected_return",
                aggfunc="mean",
            )
            .sort_index()
        )

        mean_abs_change = wide.diff().abs().mean(axis=1, skipna=True)

        rows.append(pd.DataFrame({
            "model": model,
            "rebalance_date": mean_abs_change.index,
            "mean_abs_view_change": mean_abs_change.values,
        }))

    return pd.concat(rows, ignore_index=True)


view_change_ts = compute_mean_abs_view_change(views)

view_change_summary = (
    view_change_ts
    .groupby("model")
    .agg(
        n_months=("mean_abs_view_change", "count"),
        avg_mean_abs_view_change=("mean_abs_view_change", "mean"),
        median_mean_abs_view_change=("mean_abs_view_change", "median"),
    )
    .reset_index()
)

display(view_change_summary)

view_change_pivot = (
    view_change_ts
    .pivot(index="rebalance_date", columns="model", values="mean_abs_view_change")
    .sort_index()
)

plt.figure(figsize=(12, 5))
for col in view_change_pivot.columns:
    plt.plot(view_change_pivot.index, view_change_pivot[col], label=col)

plt.title("Variation absolue moyenne des vues dans le temps")
plt.ylabel("Variation absolue moyenne")
plt.xlabel("Date")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

display(view_change_pivot.tail())


## 7. Chargement des poids Black-Litterman

Les fichiers attendus ont la forme :

```text
results/portfolio_risk_aversion_{value}/{model}_base_omega_1_black_litterman_market_implied_weights_tau_{TAU}.csv
```


In [ ]:
WEIGHT_MODELS = {
    "gpt54mini": "BL-GPT-5.4-mini",
    "gemma3": "BL-Gemma3",
    "qwen": "BL-Qwen",
    "llama": "BL-Llama",
}


def load_weight_file(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "Date" not in df.columns:
        raise ValueError(f"Colonne Date introuvable dans {path}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).copy()
    df = df.set_index("Date").sort_index()
    df.index = pd.DatetimeIndex(df.index).to_period("M").to_timestamp()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(axis=1, how="all")
    df = df[(df.index >= START_DT.to_period("M").to_timestamp()) & (df.index <= END_DT.to_period("M").to_timestamp())]

    return df


def load_bl_weights(results_dir: Path, tau: float | str, model_map: dict[str, str]) -> dict[str, pd.DataFrame]:
    weights = {}

    for model_key, display_name in model_map.items():
        path = results_dir / f"{model_key}_base_omega_1_black_litterman_market_implied_weights_tau_{tau}.csv"

        if not path.exists():
            print(f"Poids non trouvés pour {display_name}: {path}")
            continue

        weights[display_name] = load_weight_file(path)
        print(f"{display_name}: {path}")

    if not weights:
        raise FileNotFoundError(f"Aucun fichier de poids trouvé dans {results_dir}")

    return weights


bl_weights = load_bl_weights(RESULTS_DIR, TAU, WEIGHT_MODELS)

for name, w in bl_weights.items():
    print(name, w.shape, w.index.min(), "->", w.index.max())


## 8. Histogrammes cumulatifs des poids des portefeuilles

Pour garder les graphiques lisibles, seuls les `TOP_N_WEIGHTS` titres les plus importants en moyenne sont affichés séparément. Les autres titres sont regroupés dans `Autres`.


In [ ]:
TOP_N_WEIGHTS = 15


def prepare_stacked_weights(weights: pd.DataFrame, top_n: int = TOP_N_WEIGHTS) -> pd.DataFrame:
    w = weights.copy()

    # Nettoyage minimal.
    w = w.replace([np.inf, -np.inf], np.nan).fillna(0)
    w = w.loc[:, w.abs().sum(axis=0) > 0]

    if w.empty:
        return w

    top_cols = w.abs().mean().sort_values(ascending=False).head(top_n).index.tolist()
    plot_df = w[top_cols].copy()

    other_cols = [c for c in w.columns if c not in top_cols]
    if other_cols:
        plot_df["Autres"] = w[other_cols].sum(axis=1)

    # Les poids BL sont long-only. On renormalise pour éviter les petites erreurs numériques.
    plot_df = plot_df.clip(lower=0)
    row_sums = plot_df.sum(axis=1).replace(0, np.nan)
    plot_df = plot_df.div(row_sums, axis=0).fillna(0)

    return plot_df


def plot_stacked_weights(weights: pd.DataFrame, title: str, top_n: int = TOP_N_WEIGHTS) -> None:
    plot_df = prepare_stacked_weights(weights, top_n=top_n)

    if plot_df.empty:
        print(f"Aucun poids disponible pour {title}")
        return

    ax = plot_df.plot(kind="bar", stacked=True, figsize=(14, 6), width=0.9)

    step = max(1, len(plot_df.index) // 12)
    tick_positions = list(range(0, len(plot_df.index), step))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(
        [plot_df.index[i].strftime("%Y-%m") for i in tick_positions],
        rotation=45,
        ha="right",
    )

    plt.title(title)
    plt.ylabel("Poids cumulatif")
    plt.xlabel("Date")
    plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    plt.show()


for name, weights in bl_weights.items():
    plot_stacked_weights(
        weights,
        title=f"Histogramme cumulatif des poids dans le temps - {name}",
        top_n=TOP_N_WEIGHTS,
    )
